# 04 - Putting it together: a small agent, two ways

We have all the pieces now. Let's build the same small agent twice:

1. **By hand**, using the loop we wrote in notebook 3 with a list of tools and the four message types.
2. **With `create_agent`**, the high-level helper that ships in `langchain` v1 and is built on top of LangGraph.

Doing both back-to-back makes it obvious what `create_agent` saves you and what it hides. After this you should be able to read any agent code in the wild and know whether you would reach for the helper or write the loop yourself.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.messages import HumanMessage, ToolMessage, SystemMessage

model = init_chat_model("openai:gpt-4o-mini")

## The toy domain

We will build a tiny travel-research assistant. It has three tools:

1. `lookup_country_facts` - returns a one-line fact about a country (a fake knowledge base)
2. `currency_convert` - converts an amount between two currencies (fake fixed rates)
3. `add` - adds two numbers (so the agent can compose simple math)

All three are intentionally small. The point is the orchestration, not the tools.

In [ ]:
FACTS = {
    "japan":  "Japan uses the JPY. Tipping is uncommon and can be considered rude.",
    "france": "France uses the EUR. Most museums are closed on Mondays or Tuesdays.",
    "india":  "India uses the INR. Bargaining is expected at most markets.",
}

RATES_TO_USD = {"JPY": 0.0064, "EUR": 1.08, "INR": 0.012, "USD": 1.0}

@tool
def lookup_country_facts(country: str) -> str:
    """Look up a one-line travel fact for a country. Country should be lowercase English name like 'japan'."""
    return FACTS.get(country.lower(), f"No facts on file for {country}.")

@tool
def currency_convert(amount: float, from_currency: str, to_currency: str) -> float:
    """Convert an amount from one currency to another. Currencies are 3-letter ISO codes (USD, EUR, JPY, INR)."""
    if from_currency not in RATES_TO_USD or to_currency not in RATES_TO_USD:
        return float("nan")
    in_usd = amount * RATES_TO_USD[from_currency]
    return round(in_usd / RATES_TO_USD[to_currency], 2)

@tool
def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b

tools = [lookup_country_facts, currency_convert, add]
tools_by_name = {t.name: t for t in tools}

for t in tools:
    print(f"{t.name:>22} | {t.description}")

## Route 1: the hand-rolled loop

Here is the same loop pattern from notebook 3, generalized one more time so we can swap in any tool list and any starting question.

In [ ]:
def run_agent(user_input: str, model_with_tools, tools_by_name, system_prompt: str = None, max_steps: int = 8, verbose: bool = True):
    messages = []
    if system_prompt:
        messages.append(SystemMessage(system_prompt))
    messages.append(HumanMessage(user_input))

    for step in range(max_steps):
        ai_msg = model_with_tools.invoke(messages)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:
            if verbose:
                print(f"[step {step}] final answer")
            return ai_msg.text, messages

        if verbose:
            calls_summary = ", ".join(f"{tc['name']}({tc['args']})" for tc in ai_msg.tool_calls)
            print(f"[step {step}] tool calls: {calls_summary}")

        for tool_call in ai_msg.tool_calls:
            tool_obj = tools_by_name[tool_call["name"]]
            try:
                result = tool_obj.invoke(tool_call["args"])
            except Exception as e:
                result = f"Tool error: {e}"
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
            if verbose:
                print(f"           result : {result}")

    return "hit max_steps without finishing", messages

model_with_tools = model.bind_tools(tools)

answer, history = run_agent(
    "I'm planning a trip to Japan. Tell me one travel tip and convert 200 USD into JPY for me.",
    model_with_tools,
    tools_by_name,
    system_prompt="You are a friendly travel assistant. Be concise.",
)

print("\n=== FINAL ANSWER ===")
print(answer)

Take a moment to scroll back and read the trace. The model called `lookup_country_facts`, then `currency_convert`, then composed an answer. Two tool calls, two `ToolMessage` results, and a final synthesis. That is everything an agent does.

In [ ]:
# Inspect the message history at the end
for m in history:
    name = type(m).__name__
    extra = f" tool_calls={[tc['name'] for tc in m.tool_calls]}" if hasattr(m, "tool_calls") and m.tool_calls else ""
    print(f"{name:>15} | {m.text[:80]!r}{extra}")

## Route 2: `create_agent`

Now the same thing with the helper. Notice how much shorter it is and how the rest of the API stays familiar (`invoke`, `stream`).

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=tools,
    system_prompt="You are a friendly travel assistant. Be concise.",
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "I'm planning a trip to France. Give me a travel tip and convert 500 USD into EUR."}],
})

print("FINAL:", result["messages"][-1].text)

### What changed and what stayed the same

| Concern | Hand-rolled | `create_agent` |
|---|---|---|
| Build the loop | ~25 lines you write | one function call |
| Stop condition | `if not ai_msg.tool_calls: break` | handled |
| Tool errors | you wrap in try/except | handled with retry semantics |
| Persistence (resume across calls) | you build it | one flag (LangGraph checkpointer) |
| Branching control flow (e.g. "only call tool X if condition Y") | trivial, you write Python | requires a custom graph |
| Visibility into what's happening | you print whatever you want | LangSmith integration is the official path |

**My take**: `create_agent` is the right default for the obvious 80% case. Roll your own loop the moment you need control flow that does not fit the linear ReAct pattern (parallel branches, conditional tool sets, custom retry strategies, or a tight latency budget where you want to skip a model call when a heuristic suffices). If you can articulate why the hand-rolled loop is better for *your* case, the framework is not in your way.

## Inspecting the full agent message history

`create_agent` returns the entire conversation, including all the tool calls and tool messages. This is exactly the same list shape you would have built by hand.

In [ ]:
for m in result["messages"]:
    name = type(m).__name__
    extra = f" tool_calls={[tc['name'] for tc in m.tool_calls]}" if hasattr(m, "tool_calls") and m.tool_calls else ""
    print(f"{name:>15} | {m.text[:80]!r}{extra}")

## Streaming the agent

Just like a model or a chain, an agent supports `stream`. What you get back is *not* token-by-token text. It is a sequence of state updates, one per step the agent took. That is more useful than tokens for an agent because each step is itself a meaningful event ("the model decided to call X", "the tool returned Y").

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the travel tip for India and what is 75 plus 50?"}]},
    stream_mode="updates",
):
    for node, payload in chunk.items():
        msgs = payload.get("messages", [])
        for m in msgs:
            name = type(m).__name__
            print(f"[{node}] {name}: {m.text[:80]!r}")

**Insight on `stream_mode`**: agents support multiple stream modes (`updates` for state changes per node, `messages` for token-level streaming inside model calls, `values` for the full state after each step). For showing progress in a UI, `updates` is usually what you want. For a typewriter effect inside the *final answer only*, you would use `messages` and filter for the final chunk.

## A multi-turn conversation

Both routes (hand-rolled and `create_agent`) are stateless from the model's point of view. To carry context between user turns, you keep the message list around and append to it.

The `create_agent` API takes the full message list each turn:

In [ ]:
conversation = []

# turn 1
conversation.append({"role": "user", "content": "Tell me about Japan."})
out = agent.invoke({"messages": conversation})
conversation = out["messages"]  # the agent appends its own messages, take them all
print("AGENT:", conversation[-1].text)

# turn 2: a follow-up that only makes sense with memory of turn 1
conversation.append({"role": "user", "content": "And convert 1000 of their currency to USD."})
out = agent.invoke({"messages": conversation})
conversation = out["messages"]
print("AGENT:", conversation[-1].text)

Notice we never told the agent which currency "theirs" referred to in turn 2. It read the previous messages, knew Japan uses JPY, called `currency_convert(1000, 'JPY', 'USD')`, and answered. That is the multi-turn behavior emerging from nothing more than "keep the message list".

For longer-running apps you would not store the entire history in a Python variable. The LangGraph checkpointer (one of the things `create_agent` is built on) lets you persist the state to a database keyed by a session id. That is a notebook of its own; the mental model stays the same.

## Where to go from here

You have walked through every concept on the LangChain v1 main page:

- the universal model wrapper and the four message types (notebook 1)
- prompt templates, output parsers, and LCEL pipes (notebook 2)
- tool calling, the manual loop, and structured output (notebook 3)
- `create_agent` and the comparison between hand-rolled and helper (this notebook)

Topics worth exploring next, in roughly the order they show up in real apps:

1. **LangSmith** for tracing. Once you have an agent, you want to see every step. Set `LANGSMITH_API_KEY` and `LANGSMITH_TRACING=true` and every call shows up in the UI for free.
2. **RAG** (retrieval-augmented generation). Add a `search_docs` tool to the agent above and you have a domain-aware chatbot. The pieces are: a document loader, a text splitter, an embedding model, and a vector store.
3. **LangGraph** when you outgrow `create_agent`. Custom nodes, branching edges, persistence, human-in-the-loop interrupts. `create_agent` is one specific graph; LangGraph lets you draw any graph.
4. **Provider-specific features**: prompt caching, reasoning models, multimodal inputs. Each provider exposes a few things the standard interface does not. The integration package docs are the source of truth.

If you only remember one sentence from these four notebooks: *LangChain is the model wrapper, the message types, and the pipe operator. Everything else is patterns built on those three things.*